In [ ]:
import os
import pandas as pd
import numpy as np

import pickle

import matplotlib.pyplot as plt

from sklearn.metrics import roc_auc_score, mean_squared_error

import shap

from lightgbm import LGBMClassifier
from sklearn.pipeline import Pipeline


import warnings
warnings.filterwarnings('ignore')

# Leitura dos artefatos

In [ ]:
model = pickle.load(
    open('../models/wrapped/model_pipeline_prod.pkl', 'rb')
)
model


# Classificação

## Leitura das bases (treino e teste)

In [ ]:
df_treino = pd.read_csv(os.path.join('..', 'data', 'train_test', 'train.csv'))

print(df_treino.shape)
df_treino.head()

In [ ]:
df_teste = pd.read_csv(os.path.join('..', 'data', 'train_test', 'test.csv'))

print(df_teste.shape)
df_teste.head()

In [ ]:
seletor = pickle.load(
    open(os.path.join('..', 'models', 'encoders', 'seletor_2.pkl'), 'rb')
)

## Scoring

In [ ]:
df_treino['score'] = (model.predict_proba(df_treino)[:,1]*1000).astype(int)
df_teste['score'] = (model.predict_proba(df_teste)[:,1]*1000).astype(int)

## Evaluation

In [ ]:
roc_auc_score(
    df_treino['y'],
    df_treino['score']/1000
)

In [ ]:
roc_auc_score(
    df_teste['y'],
    df_teste['score']/1000
)

## Análise de resultados

In [ ]:
print(hasattr(model[-1], 'booster_'))  # True se o modelo estiver treinado


In [ ]:
booster = model[-1].booster_
importance = booster.feature_importance(importance_type='split')  

df_imp = pd.DataFrame({
    'feature': seletor.features,
    'imp': importance
}).sort_values(by='imp', ascending=False)


In [ ]:
plt.figure(figsize=(10, 6))

plt.barh(df_imp['feature'], df_imp['imp'], color = 'skyblue')
plt.xlabel('Importância')
plt.ylabel('Features')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.grid(axis = 'x', linestyle = '--', alpha = 0.6)

plt.tight_layout()
plt.show()

In [ ]:
df_treino_encoded = pd.read_csv(os.path.join('..', 'data', 'train_test', 'train_encoded.csv'))
sample_train = df_treino_encoded[seletor.features].sample(frac = 0.3, random_state=98)

In [ ]:
explainer = shap.Explainer(model[-1], sample_train)
shap_values = explainer(sample_train)

plt.figure(figsize = (10, 6))
shap.summary_plot(shap_values, sample_train)